# RAG Backdoor — dual-surface EVAL on Colab (uses the already-trained adapter)

Runs each attack (trigger / conflict / soft) under three defense conditions
(none / RFC / EllipticEnvelope) against the backdoored model, and reports ASR,
clean accuracy, targeted clean accuracy, and faithfulness.

**Runtime → Change runtime type → T4 GPU.**

You need two files (both already on your laptop):
1. `rag-backdoor-defense.tar.gz` — the **updated** code (re-download from ki-030 `/home/8e4d/`).
2. `backdoor_adapter.zip` — the adapter you downloaded after training.

## Cell 1 — GPU + install deps

In [ ]:
!nvidia-smi --query-gpu=name,memory.free --format=csv
%pip install -q -U "transformers>=4.40.0" "peft>=0.10.0" "bitsandbytes>=0.43.0" \
    "accelerate>=0.29.0" "datasets>=2.19.0" "sentence-transformers>=2.7.0" \
    faiss-cpu "rouge-score>=0.1.2" "scikit-learn>=1.4.0" pyyaml
print('deps installed')

## Cell 2 — Upload the updated code tarball and extract

In [ ]:
import os
from google.colab import files
print('Select rag-backdoor-defense.tar.gz ...')
up = files.upload()
os.makedirs('/content/proj', exist_ok=True)
!tar -xzf "{next(iter(up))}" -C /content/proj
%cd /content/proj
!ls src scripts

## Cell 3 — Upload the adapter zip and unpack it into results/

In [ ]:
from google.colab import files
print('Select backdoor_adapter.zip ...')
up = files.upload()
!mkdir -p results && unzip -oq "{next(iter(up))}" -d results
!ls results/backdoor_adapter/adapter_config.json results/backdoor_adapter/adapter_model.safetensors

## Cell 4 — Hugging Face login (gated base model)

In [ ]:
from huggingface_hub import login
from getpass import getpass
login(getpass('HF token (hf_...): '))
print('logged in')

## Cell 5 — Run the dual-surface evaluation
~15 min on a T4 (corpus encodes once on CPU, then ~600 generations). RFC threshold
0.03 is ROC-optimal. Targets the first 15 questions for the query-relevant attacks.
Reports ASR, clean accuracy, targeted accuracy, faithfulness, and Rank Poisoning Score.

In [ ]:
!USE_TF=0 python scripts/eval_dual_surface.py \
    --adapter results/backdoor_adapter \
    --samples 50 --n-poison 15 --rfc-threshold 0.03 \
    --out results/dual_surface_eval.json

## Cell 6 — Results table + download JSON

In [ ]:
import json, pandas as pd
from google.colab import files
res = json.load(open('results/dual_surface_eval.json'))['results']
rows = []
for attack, conds in res.items():
    for cond, m in conds.items():
        rows.append({'attack': attack, 'defense': cond,
                     'ASR': m.get('asr'),
                     'CleanAcc': m.get('clean_accuracy_rougeL'),
                     'TargetedAcc': m.get('targeted_clean_accuracy'),
                     'Faithful': m.get('faithfulness'),
                     'RankPoison': m.get('rank_poisoning_score'),
                     'poison_in_ctx': m.get('avg_poison_in_context'),
                     'rfc_flagged': m.get('total_rfc_flagged'),
                     'blocked@ingest': m.get('n_poison_blocked_at_ingestion')})
import IPython.display as d; d.display(pd.DataFrame(rows))
files.download('results/dual_surface_eval.json')